# CTM-LLM: Visualizing the Continuous Thought Machine
## — 探索人造大脑的神经动力学

本笔记本展示 CTM 如何像生物大脑一样工作：神经元通过迭代（"思考嘀嗒"）逐步演化、同步放电、形成记忆轨迹。

> **灵感来源**: Continuous Thought Machines (Sakana AI, 2025)
> 本 CTM-LLM 将其移植为因果语言模型，保留了内部迭代循环和神经元级记忆。

### 目录
1. 加载模型
2. 内部状态追踪
3. 可视化 1：神经元放电热图
4. 可视化 2：同步化热图
5. 可视化 3：状态记忆轨迹
6. 可视化 4：神经活动 GIF 动画
7. 可视化 5：Decay 率分析
8. 可视化 6：浅层 vs 深层
9. 生物性总结

In [16]:
import os, sys, math, uuid
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import seaborn as sns
from IPython.display import HTML, display
from transformers import AutoTokenizer

sys.path.insert(0, os.path.abspath('.'))
from model.config import CTMLLMConfig
from model.model_ctm_llm import CTMForCausalLM

sns.set_theme(style='white')
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

Using device: cuda:0


In [17]:
WEIGHT = './out/ctm_llm_512_resume.pth'

cfg = CTMLLMConfig(
    vocab_size=6400, hidden_size=512, num_hidden_layers=8,
    d_model=256, d_input=128, iterations=3, memory_length=5, heads=4,
    n_synch_out=256, n_synch_action=256, synapse_depth=2,
)

model = CTMForCausalLM(cfg).to(device)
ckpt = torch.load(WEIGHT, map_location=device, weights_only=False)
state = ckpt['model'] if 'model' in ckpt else ckpt
model.load_state_dict(state, strict=False)
model.eval()
print(f'Loaded: {WEIGHT}')
print(f'Iterations (ticks): {cfg.iterations}')
print(f'Neurons (d_model):  {cfg.d_model}')
print(f'Memory length:      {cfg.memory_length}')
print(f'Layers:             {cfg.num_hidden_layers}')
print(f'Synch action:       {cfg.n_synch_action}')
print(f'Synch out:          {cfg.n_synch_out}')

Loaded: ./out/ctm_llm_512_resume.pth
Iterations (ticks): 3
Neurons (d_model):  256
Memory length:      5
Layers:             8
Synch action:       256
Synch out:          256


In [18]:
tokenizer = AutoTokenizer.from_pretrained('./model_tokenizer')

PROMPT = 'Give three tips for staying healthy.'
print(f'Prompt: {PROMPT}')

messages = [{'role': 'user', 'content': PROMPT}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
ids = tokenizer.encode(text)
input_ids = torch.tensor([ids], dtype=torch.long, device=device)
print(f'Input tokens: {len(ids)}')
for i, tid in enumerate(ids):
    print(f'  [{i:2d}] {tid:>5} {repr(tokenizer.decode([tid]))}')

Prompt: Give three tips for staying healthy.
Input tokens: 30
  [ 0]     1 '<|im_start|>'
  [ 1]   832 'us'
  [ 2]   311 'er'
  [ 3]   234 '\n'
  [ 4]    74 'G'
  [ 5]   884 'ive'
  [ 6]  3785 ' three'
  [ 7]   297 ' t'
  [ 8]  1367 'ip'
  [ 9]   118 's'
  [10]   503 ' for'
  [11]   580 ' st'
  [12]   655 'ay'
  [13]   350 'ing'
  [14]  3983 ' health'
  [15]   124 'y'
  [16]    49 '.'
  [17]     2 '<|im_end|>'
  [18]   234 '\n'
  [19]     1 '<|im_start|>'
  [20]  1388 'ass'
  [21]   570 'ist'
  [22]   811 'ant'
  [23]   234 '\n'
  [24]    25 '<think>'
  [25]   234 '\n'
  [26]   234 '\n'
  [27]    26 '</think>'
  [28]   234 '\n'
  [29]   234 '\n'


## 前向传播 + 内部状态追踪

使用 `forward_track` 捕获 CTMBlock 循环中每个迭代的内部变量：
- `post_activations`: 突触后活动（类动作电位）
- `pre_activations`: 突触前活动
- `sync_action`: 动作同步化
- `state_trace`: 滑动窗口记忆（类工作记忆）
- `decay_action/decay_output`: 学习到的生物衰减率

In [19]:
with torch.amp.autocast('cuda', dtype=torch.bfloat16):
    tracking, logits, probs = model.forward_track(input_ids)

print('Layers tracked:', list(tracking.keys()))
layer0 = tracking['layer_0']
T = layer0['pre_activations'].shape[1]
print(f'\nlayer_0 data shapes:')
for k, v in layer0.items():
    if isinstance(v, np.ndarray):
        print(f'  {k:25s} {v.shape}')
    else:
        print(f'  {k:25s} type={type(v).__name__}')

Layers tracked: ['layer_0', 'layer_1', 'layer_2', 'layer_3', 'layer_4', 'layer_5', 'layer_6', 'layer_7']

layer_0 data shapes:
  pre_activations           (3, 30, 256)
  post_activations          (3, 30, 256)
  sync_action               (3, 30, 256)
  sync_output               (1, 30, 256)
  attention_weights         type=list
  state_trace               (3, 30, 256, 5)
  decay_action              (256,)
  decay_output              (256,)


---
## 1. 神经元放电动态

> **生物学类比**: 每个迭代如同一次"思考脉冲"，神经元集群的集体活动模式在高维空间中演化。

当前迭代数=3，我们用热图展示所有神经元在不同迭代时刻的激活强度。每列是一次迭代，每行是一个神经元。

In [28]:
N_SHOW = 120
tp = min(T - 1, 10)
post = layer0['post_activations']  # (3, T, 256)

data = post[:, tp, :N_SHOW].T
v = max(abs(data.min()), abs(data.max()))

fig, ax = plt.subplots(figsize=(8, 10))
im = ax.imshow(data, aspect='auto', cmap='RdBu_r', vmin=-v, vmax=v, interpolation='nearest')
ax.set_xticks(range(3))
ax.set_xticklabels(['Tick 0', 'Tick 1', 'Tick 2'])
ax.set_ylabel('Neuron Index')
ax.set_title(f'Neuron Activity @ token[{tp}]  (first {N_SHOW} neurons)', fontsize=13)
plt.colorbar(im, ax=ax, label='Activation', shrink=0.8)
plt.tight_layout()
plt.savefig('out/neural_firing_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: out/neural_firing_heatmap.png')

# Show neurons with biggest change
print('\n=== 变化最大的 10 个神经元 ===')
deltas = [(n, post[:, tp, n].max() - post[:, tp, n].min()) for n in range(cfg.d_model)]
deltas.sort(key=lambda x: -x[1])
for n, d in deltas[:10]:
    vals = post[:, tp, n]
    print(f'  neuron[{n:3d}]: [{vals[0]:+.4f}, {vals[1]:+.4f}, {vals[2]:+.4f}]  Δ={d:.4f}')

Saved: out/neural_firing_heatmap.png

=== 变化最大的 10 个神经元 ===
  neuron[ 57]: [-0.5212, +0.2492, +3.7980]  Δ=4.3192
  neuron[ 85]: [-0.7117, -1.7431, -4.1985]  Δ=3.4868
  neuron[227]: [+0.0483, +2.0592, +2.8143]  Δ=2.7661
  neuron[152]: [-0.5609, -1.9381, -2.8461]  Δ=2.2853
  neuron[186]: [+1.3885, +2.0444, +2.9450]  Δ=1.5565
  neuron[217]: [+0.0280, +0.6099, +1.5430]  Δ=1.5150
  neuron[ 38]: [-0.2093, -1.2392, -1.6316]  Δ=1.4223
  neuron[189]: [-0.2222, +0.2662, -0.8270]  Δ=1.0932
  neuron[147]: [-0.0558, +0.8685, +0.2445]  Δ=0.9243
  neuron[145]: [+0.1165, -0.7673, -0.1986]  Δ=0.8838


---
## 2. 同步化热图

> **生物学类比**: 不同神经元对的同步放电——类似脑电波相干性 (coherence)。红=正同步，蓝=负同步。

In [29]:
n_synch = layer0['sync_action'].shape[-1]
n_s = min(64, n_synch)
sync_data = layer0['sync_action'][:, tp, :n_s].T

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sync_data, aspect='auto', cmap='RdYlBu_r', vmin=-1, vmax=1, interpolation='nearest')
ax.set_xticks(range(3))
ax.set_xticklabels([f'Tick 0', 'Tick 1', 'Tick 2'])
ax.set_xlabel('Thinking Tick')
ax.set_ylabel('Synch Pair')
ax.set_title(f'Action Synchronization @ token[{tp}] (Pairs 0-{n_s-1})', fontsize=13)
plt.colorbar(im, ax=ax, label='Sync Strength', shrink=0.8)
plt.tight_layout()
plt.savefig('out/synch_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

**多 token 对比**: 看同一层不同 token 位置的同步化模式差异。

In [30]:
n_tok = min(8, T)
n_s = min(32, n_synch)
sync = layer0['sync_action']

fig, axes = plt.subplots(1, n_tok, figsize=(3*n_tok+1, 4))
if n_tok == 1: axes = [axes]
for t_idx in range(n_tok):
    axes[t_idx].imshow(sync[:, t_idx, :n_s].T, aspect='auto', cmap='RdYlBu_r', vmin=-1, vmax=1)
    axes[t_idx].set_xticks(range(3))
    axes[t_idx].set_xticklabels(['T0','T1','T2'])
    axes[t_idx].set_title(f'tok[{t_idx}]')
plt.suptitle('Synchronization across tokens (layer_0)', fontsize=13)
plt.tight_layout()
plt.savefig('out/synch_tokens.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. 状态记忆轨迹

> **生物学类比**: 每个神经元维护一个长度为 5 的滑动窗口记忆（短期突触可塑性），记录最近的 pre-activation 历史。

In [31]:
st = layer0['state_trace']  # (3, T, 256, 5)
mem_len = st.shape[-1]
n_neurons = 16

fig, axes = plt.subplots(4, 4, figsize=(14, 10))
axes = axes.flatten()
for n_idx in range(n_neurons):
    ax = axes[n_idx]
    for it in range(3):
        trace = st[it, tp, n_idx, :]
        ax.plot(range(mem_len), trace, marker='o', label=f'T{it}', alpha=0.7)
    ax.set_title(f'Neuron {n_idx}', fontsize=9)
    ax.axhline(0, color='gray', lw=0.5, ls='--')

fig.suptitle(f'Memory Traces (window={mem_len}) @ token[{tp}]', fontsize=14)
fig.tight_layout()
plt.savefig('out/memory_traces.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. 神经活动 GIF 动画

动画展示神经元活动如何随迭代变化。灰色列 = 之前迭代的快照，高亮列 = 当前活跃迭代。

In [24]:
def animate_neural(tracking, layer_k='layer_0', token_pos=10):
    post = tracking[layer_k]['post_activations']
    n_show = min(80, post.shape[-1])
    v = max(abs(post.min()), abs(post.max()))

    fig, axes = plt.subplots(1, 3, figsize=(10, 6))
    ims = []
    for i in range(3):
        ax = axes[i]
        im = ax.imshow(post[i:i+1, token_pos, :n_show].T, aspect='auto',
                       cmap='RdBu_r', vmin=-v, vmax=v, interpolation='nearest')
        ax.set_title(f'Tick {i}')
        ax.set_ylabel('Neuron' if i == 0 else '')
        ims.append(im)
    fig.suptitle(f'{layer_k} @ token[{token_pos}]', fontsize=14)
    plt.tight_layout()

    def update(frame):
        for i in range(3):
            alpha = 1.0 if i == frame else 0.2
            data = post[i:i+1, token_pos, :n_show].T
            ims[i].set_array(data)
            ims[i].set_alpha(alpha)
            axes[i].set_title(f'Tick {i}' + (' ◀' if i == frame else ''))
        return ims

    ani = animation.FuncAnimation(fig, update, frames=3, interval=1200, blit=False)
    plt.close(fig)
    return ani

ani = animate_neural(tracking, 'layer_0', token_pos=tp)
ani.save('out/neural_firing_animated.gif', writer='pillow', fps=1)
print('Saved: out/neural_firing_animated.gif')
display(HTML(ani.to_jshtml()))

Saved: out/neural_firing_animated.gif


---
## 5. Decay 率的生物学分析

> **突触衰减**: $r = e^{-param}$。$r \to 1$ = 慢遗忘（长时记忆），$r \to 0$ = 快遗忘（短时记忆）。CTM 学习到每个同步化对的遗忘速率。

In [25]:
d_a = layer0['decay_action']
d_o = layer0['decay_output']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, d, lab, col in zip(axes, [d_a, d_o], ['Action Decay', 'Output Decay'], ['steelblue', 'coral']):
    ax.hist(d, bins=50, color=col, alpha=0.7, edgecolor='white')
    ax.set_title(f'{lab}  n={len(d)}')
    ax.set_xlabel('Decay rate r')
    ax.set_ylabel('Count')
    ax.axvline(d.mean(), color='red', ls='--', label=f'mean={d.mean():.4f}')
    ax.legend()

plt.suptitle('Biological Synaptic Decay Rates', fontsize=14)
plt.tight_layout()
plt.savefig('out/decay_rates.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Action decay range: [{d_a.min():.4f}, {d_a.max():.4f}]  mean={d_a.mean():.4f}')
print(f'Output decay range: [{d_o.min():.4f}, {d_o.max():.4f}]  mean={d_o.mean():.4f}')

Action decay range: [0.3755, 0.9996]  mean=0.7713
Output decay range: [0.3855, 0.9999]  mean=0.8937


---
## 6. 浅层 vs 深层对比

Layer 0（浅层，处理低级特征）vs Layer 7（深层，处理高级语义）。

In [26]:
n_show = min(80, cfg.d_model)
post0 = tracking['layer_0']['post_activations']
post7 = tracking['layer_7']['post_activations']
v = max(abs(post0.min()), abs(post0.max()), abs(post7.min()), abs(post7.max()))

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for i in range(3):
    axes[0, i].imshow(post0[i, :, :n_show].T, aspect='auto', cmap='RdBu_r', vmin=-v, vmax=v, interpolation='nearest')
    axes[0, i].set_title(f'Layer 0 Tick {i}')
    axes[0, i].set_ylabel('Neuron')
    axes[1, i].imshow(post7[i, :, :n_show].T, aspect='auto', cmap='RdBu_r', vmin=-v, vmax=v, interpolation='nearest')
    axes[1, i].set_title(f'Layer 7 Tick {i}')
    axes[1, i].set_xlabel('Token')
    axes[1, i].set_ylabel('Neuron')

plt.suptitle('Shallow vs Deep Layer Activity', fontsize=14)
plt.tight_layout()
plt.savefig('out/layer_depth.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. 生物性总结

| 生物大脑 | CTM | 可视化 |
|---|---|---|
| 🧠 动作电位 | Post-activation | 神经元活动热图 |
| 🔄 神经同步 | Action/Output Synch | 同步化热图 |
| 💾 工作记忆 | State Trace (mem_len=5) | 记忆轨迹图 |
| ⏳ 突触衰减 | $r = e^{-param}$ | Decay 分布图 |
| 📊 皮层分层 | 8 层 CTM | 浅层 vs 深层 |

**核心洞察**: CTM 将生物大脑的多个关键机制整合进一个可微分网络——更像是"人造皮层柱"而非传统 Transformer。

In [27]:
print('=== 所有可视化文件 ===')
for f in ['neural_firing_heatmap.png', 'synch_heatmap.png', 'synch_tokens.png',
          'memory_traces.png', 'neural_firing_animated.gif', 'decay_rates.png', 'layer_depth.png']:
    path = f'out/{f}'
    print(f'  {path} ({os.path.getsize(path)/1024:.0f} KB)' if os.path.exists(path) else f'  {path} (missing)')

=== 所有可视化文件 ===
  out/neural_firing_heatmap.png (47 KB)
  out/synch_heatmap.png (58 KB)
  out/synch_tokens.png (40 KB)
  out/memory_traces.png (499 KB)
  out/neural_firing_animated.gif (71 KB)
  out/decay_rates.png (54 KB)
  out/layer_depth.png (62 KB)
